## LangFuse 실습

관측 도구에 실행 기록을 보내고, 같은 `run_id` 로 우리 DB 에서도 찾아본다.

In [ ]:
import os
import pathlib
import sys

# 노트북이 어느 위치에서 실행되든 backend/app 이 들어 있는 폴더(hanwha-agent)를 찾아 루트로 삼는다
# - 폴더 이름(day01)에 기대지 않으므로 다른 날짜 노트북에 복사해도 그대로 쓸 수 있다
here = pathlib.Path.cwd().resolve()
candidates = [here, *here.parents, here / "hanwha-agent"]
ROOT = next((p for p in candidates if (p / "backend" / "app").is_dir()), None)
if ROOT is None:
    raise RuntimeError(f"hanwha-agent 루트를 찾지 못했습니다. 현재 위치: {here}")

os.chdir(ROOT)                                  # 상대경로(.env 등)의 기준
SANDBOX = ROOT / "sandbox" / "w4" / "day01"

# app 패키지를 import 할 수 있게 backend 를 모듈 검색 경로 맨 앞에 넣는다
# - os.chdir 만으로는 import 경로가 바뀌지 않는다
BACKEND = str(ROOT / "backend")
if BACKEND not in sys.path:
    sys.path.insert(0, BACKEND)

print("프로젝트 루트  :", ROOT)

- 설정 확인

In [ ]:
import importlib

importlib.invalidate_caches()

import app.core.config as config_module

config_module = importlib.reload(config_module)
config_module.get_settings.cache_clear()          # @lru_cache 가 쥐고 있는 옛 설정을 버린다
settings = config_module.get_settings()

print("관측 사용   :", settings.langfuse_enabled)
print("보낼 곳     :", settings.langfuse_host)
print("public key  :", config_module.mask(settings.langfuse_public_key))
print("secret key  :", config_module.mask(
    settings.langfuse_secret_key.get_secret_value() if settings.langfuse_secret_key else None
))

# 키가 없어도 노트북은 끝까지 돈다. 관측 부분만 건너뛴다
HAS_KEYS = bool(settings.langfuse_public_key) and settings.langfuse_secret_key is not None
print()
print("키 준비 여부 :", HAS_KEYS)
if not HAS_KEYS:
    print("  .env 의 LANGFUSE_PUBLIC_KEY / LANGFUSE_SECRET_KEY 를 채우면 아래 셀이 실제로 전송합니다.")

- v2 방식으로 트레이스 한 건 보내기 (화면 반영까지 몇 분 걸린다)

In [ ]:
from datetime import datetime, timezone

import anthropic
from langfuse import Langfuse

# 연습 질문과 프롬프트
QUESTION = "부산 출장 숙박비 한도가 얼마인가요?"
SYSTEM = "너는 사내 규정 질의응답 도우미다. 근거가 없으면 없다고 말한다."

if not HAS_KEYS:
    print("키가 없어 이 셀은 건너뜁니다.")
else:
    # 랭퓨즈 객체 생성
    lf = Langfuse(
        public_key=settings.langfuse_public_key,
        secret_key=settings.langfuse_secret_key.get_secret_value(),
        host=settings.langfuse_host,
    )
    print("보낼 곳 :", settings.langfuse_host)

    # 키가 이 주소에서 통하는 키인지 먼저 확인한다
    # - 이걸 건너뛰면 아래에서 조용히 전송만 실패하고 화면에 아무것도 안 뜬다
    try:
        lf.auth_check()
    except Exception as e:
        print("인증 실패 :", type(e).__name__)
        print("  ① 키를 발급한 리전과 LANGFUSE_HOST 가 같은가 (jp.cloud · cloud · us.cloud)")
        print("  ② pk-lf- 와 sk-lf- 를 서로 바꿔 붙이지 않았는가")
        print("  ③ 키 앞뒤에 공백이나 따옴표가 붙지 않았는가")
        print("  ④ localhost 라면 컨테이너가 떠 있는가 (docker compose ps)")
        raise SystemExit
    print("인증     : 통과")

    # 트레이스 하나와 그 안에 제네레이션 하나
    # - trace       : 사용자 요청 1건
    # - generation  : 그 안의 모델 호출 1회 (토큰·모델명이 붙는 특별한 observation)
    trace = lf.trace(
        name="trace-hello",
        input=QUESTION,
        tags=["w04d01", "hello"],
        metadata={"host": settings.langfuse_host},
    )
    generation = trace.generation(
        name="claude",
        model=settings.llm_model,
        input=[{"role": "system", "content": SYSTEM},
               {"role": "user", "content": QUESTION}],
        start_time=datetime.now(timezone.utc),
    )

    # 클로드 실제 호출
    key = settings.anthropic_api_key
    try:
        if key is None:
            raise RuntimeError(".env 의 ANTHROPIC_API_KEY 가 비어 있습니다")
        client = anthropic.Anthropic(api_key=key.get_secret_value())
        resp = client.messages.create(
            model=settings.llm_model,
            max_tokens=settings.max_tokens,
            system=SYSTEM,
            messages=[{"role": "user", "content": QUESTION}],
        )
        answer = "".join(b.text for b in resp.content if b.type == "text")
        generation.end(
            output=answer,
            usage_details={"input": resp.usage.input_tokens,
                           "output": resp.usage.output_tokens},
        )
        trace.update(output=answer)
        print("Claude   :", answer[:60].replace("\n", " "), "...")
        print("토큰     : 입력", resp.usage.input_tokens, "· 출력", resp.usage.output_tokens)
    except Exception as e:
        # 실패도 기록한다 — 실패한 호출이 기록에서 빠지면 성공률을 계산할 수 없다
        generation.end(level="ERROR", status_message=f"{type(e).__name__}: {e}")
        trace.update(output=None)
        print("Claude   : 호출 실패 -", type(e).__name__)
        print("           그래도 트레이스는 보냅니다. 화면에서 빨간 ERROR 로 보입니다.")

    # 보내기
    # - SDK 는 성능을 위해 기록을 모아뒀다가 background 로 보낸다
    #   노트북·스크립트는 그냥 끝나버리므로 flush() 를 부르지 않으면 아무것도 안 올라간다
    lf.flush()

    # 화면 주소 만들기 (옵션)
    base = settings.langfuse_host.rstrip("/")
    try:
        project_id = lf.client.projects.get().data[0].id
        print("트레이스 :", f"{base}/project/{project_id}/traces/{trace.id}")
    except Exception:
        print("트레이스 :", trace.id, "(화면 왼쪽 Tracing → Traces 에서 찾으세요)")

- v4 계열 문법 (참고만, 실행하지 않음)

`langfuse` 2.x 와 4.x 는 API 가 다르다. 4.x 는 `trace()` / `generation()` 대신
`start_as_current_observation()` 컨텍스트 매니저로 계층을 만든다.
버전을 올릴 때 이 부분을 고쳐야 한다.

```python
lf = Langfuse(
    public_key=settings.langfuse_public_key,
    secret_key=settings.langfuse_secret_key.get_secret_value(),
    host=settings.langfuse_host,
)

with lf.start_as_current_observation(as_type="span", name="trace-hello", input=QUESTION) as trace:
    with lf.start_as_current_observation(
        as_type="generation",
        name="claude",
        model=settings.llm_model,
        input=[{"role": "system", "content": SYSTEM},
               {"role": "user", "content": QUESTION}],
    ) as generation:
        # Claude 호출
        generation.update(output=answer)
    trace.update(output=answer)

lf.flush()
```

- 어댑터 확인 : 꺼져 있어도 아무것도 깨지지 않는다

In [ ]:
import importlib

importlib.invalidate_caches()

from app.integrations import langfuse_client

# LANGFUSE_ENABLED=false 이면 get_client() 가 None 을 돌려준다
# - 예외가 아니라 None 이다. 부르는 쪽이 if 한 줄로 넘어갈 수 있게 만든 규약이다
print("get_client() :", langfuse_client.get_client())

# 관측이 꺼져 있어도 with 블록은 그대로 지나간다 (handle 만 None)
with langfuse_client.trace("ask", run_id="RUN-1234") as handle:
    print("trace() 의 handle :", handle)
    print(f"with 블록 안 계산 : 3 + 4 = {3 + 4}")

# 점수도 마찬가지. 꺼져 있으면 아무 일도 일어나지 않는다
langfuse_client.score("RUN-1234", "golden_pass", 1.0)
print("score() 호출 : 예외 없음")

- ask() 가 runs 표에 행을 남기는지

In [ ]:
import importlib

importlib.invalidate_caches()

import app.integrations.factory as factory
from sqlalchemy import select

from app.db.session import session_scope
from app.integrations.ports import LLMResult
from app.models import Run
from app.services import chat_service

REPLY = ('{"answer": "부산 출장 숙박비는 1박 7만원 이내입니다.", '
         '"sources": [{"doc_id": "DOC-HR-014", "title": "국내출장 여비 규정", '
         '"version": "v2.0", "locator": "제12조(숙박비) · p.6"}], '
         '"enough_evidence": true}')


# 실제 모델을 부르지 않는 가짜 어댑터
class StubLLM:
    name = "stub"

    def answer(self, *, question: str, contexts: list[dict], user: dict) -> LLMResult:
        return LLMResult(text=REPLY, model="claude-haiku-4-5",
                         input_tok=1200, output_tok=300, cost_krw=3.8, latency_ms=900)


original = factory.get_llm
try:
    factory.get_llm = lambda: StubLLM()
    # run_id 를 주지 않았다 → 서비스가 next_run_id() 로 직접 만든다
    out = chat_service.ask(question="부산 출장 숙박비 한도가 얼마인가요?")
finally:
    factory.get_llm = original

print("ask() 가 돌려준 run_id :", out.run_id)
print()

# 같은 번호로 DB 에서 찾을 수 있어야 한다 — 이게 run_id 를 먼저 만드는 이유다
with session_scope() as session:
    run = session.scalars(select(Run).where(Run.id == out.run_id)).one()
    print("runs 표에서 찾은 행")
    print("  id         :", run.id)
    print("  user_id    :", run.user_id, "      (김민준 · 인프라사업부 2팀)")
    print("  status     :", run.status)
    print("  mode       :", run.mode)
    print("  latency_ms :", run.latency_ms)
    print("  answer     :", run.answer[:16] + "...")
    print("  sources    :", run.sources)

- GET /api/v1/chat/runs/{run_id}

In [ ]:
from fastapi.testclient import TestClient

from app.main import app

client = TestClient(app)

# 방금 만든 실행 기록을 API 로 조회
r_ok = client.get(f"/api/v1/chat/runs/{out.run_id}")
print("status code :", r_ok.status_code)
print("question    :", r_ok.json()["question"])
print("status      :", r_ok.json()["status"])
print("created_at  :", r_ok.json()["created_at"])

# 없는 번호는 404 — 서비스가 던진 NotFound 를 main.py 전역 핸들러가 바꿔 내보낸다
r_no = client.get("/api/v1/chat/runs/RUN-9999")
print()
print("RUN-9999 status code :", r_no.status_code)
print("RUN-9999 body        :", r_no.json())